# PROYECTO 10

Generación de modelo predictivo de recomendación de planes para la empresa Megaline.


## Evaluación inicial al juego de datos disponible:


In [1]:
#Carga de librerias requeridas:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

#Recolección de datos:
df = pd.read_csv('/datasets/users_behavior.csv')
print(df.head(10))
print()
print('Forma del juego de datos (Filas, Columnas): ', df.shape)
print()
print(df.info())
print()
print('Valores duplicados: ', df.duplicated().sum())
print()
print(df.describe())
df[['calls', 'messages']] = df[['calls', 'messages']].astype(int)
print()
print(df.info())


   calls  minutes  messages   mb_used  is_ultra
0   40.0   311.90      83.0  19915.42         0
1   85.0   516.75      56.0  22696.96         0
2   77.0   467.66      86.0  21060.45         0
3  106.0   745.53      81.0   8437.39         1
4   66.0   418.74       1.0  14502.75         0
5   58.0   344.56      21.0  15823.37         0
6   57.0   431.64      20.0   3738.90         1
7   15.0   132.40       6.0  21911.60         0
8    7.0    43.39       3.0   2538.67         1
9   90.0   665.41      38.0  17358.61         0

Forma del juego de datos (Filas, Columnas):  (3214, 5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3214 entries, 0 to 3213
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   calls     3214 non-null   float64
 1   minutes   3214 non-null   float64
 2   messages  3214 non-null   float64
 3   mb_used   3214 non-null   float64
 4   is_ultra  3214 non-null   int64  
dtypes: float64(4), int64(1)
memory 

### Observaciones

Las principales características del juego de datos es el siguiente:
- En total se tienen registros para 3214 usuarios.
- Se realiza revisión y no se encuentran datos ausentes o duplicados.
- Un análisis preliminar permite identificar que cerca de 31% de los usuarios hacen parte del plan Ultra y el restante al plan Smart.
- Para efectos estéticos, se convierten los datos de calls y messages de formato float64 a tipo int

## Segmentación del juego de datos:


In [2]:
#Segmentación del juego de datos:
#Entrenamiento:

df_train, df_split = train_test_split(df, test_size=0.4, random_state=792604)
df_valid, df_test = train_test_split(df_split, test_size=0.5, random_state=792604)
print(df_train.info())
print()
print(df_valid.info())
print()
print(df_test.info())

<class 'pandas.core.frame.DataFrame'>
Int64Index: 1928 entries, 1159 to 1205
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   calls     1928 non-null   int64  
 1   minutes   1928 non-null   float64
 2   messages  1928 non-null   int64  
 3   mb_used   1928 non-null   float64
 4   is_ultra  1928 non-null   int64  
dtypes: float64(2), int64(3)
memory usage: 90.4 KB
None

<class 'pandas.core.frame.DataFrame'>
Int64Index: 643 entries, 3094 to 2743
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   calls     643 non-null    int64  
 1   minutes   643 non-null    float64
 2   messages  643 non-null    int64  
 3   mb_used   643 non-null    float64
 4   is_ultra  643 non-null    int64  
dtypes: float64(2), int64(3)
memory usage: 30.1 KB
None

<class 'pandas.core.frame.DataFrame'>
Int64Index: 643 entries, 374 to 2372
Data columns (total 5 columns):
 #   Column    N

### Observaciones:

Al no disponer de mas juegos de datos, se hace una segmentación de los datos para el entrenamiento del modelo de la siguiente manera:
- 60% de los datos se destinarán para entrenamiento del modelo.
- 20% de los datos se utilizarán en la validación del modelo.
- 20% de los datos restantes se usarán en la prueba del modelo.



## Definición de Característica objetivo y demas:

In [3]:
#Partición set de entrenamiento:
feature_train = df_train.drop(['is_ultra'], axis=1)
target_train = df_train['is_ultra']

#Partición set de validación:
feature_valid = df_valid.drop(['is_ultra'], axis=1)
target_valid = df_valid['is_ultra']

#Partición set de prueba:
feature_test = df_test.drop(['is_ultra'], axis=1)
target_test = df_test['is_ultra']

### Observaciones:
Se define como característica objetivo la columna 'is_ultra' que determinará que plan recomendar de acuerdo a sus consumos de llamadas, mensajes, minutos utilizados y mb consumidos.

## Modelo de Árbol de decisión:

In [6]:
#Definición hiperparámetros iniciales:
best_depth_dt = 0
best_acc_dt = 0

#Aplicación del modelo
for depth in range(1,11):
    model_dt = DecisionTreeClassifier(random_state=792604, max_depth=depth)
    model_dt.fit(feature_train, target_train)
    prediction_dt = model_dt.predict(feature_valid)
    acc_dt = accuracy_score(target_valid, prediction_dt)

    #Definición de los mejores parámetros
    if acc_dt > best_acc_dt:
        best_depth_dt = depth
        best_acc_dt = acc_dt

#Datos finales
print('Datos para modelo de Árbol de decisión:')
print()
print('profundidad del árbol:', best_depth_dt)
print()
print('Exactitud del módelo:', best_acc_dt)
if best_acc_dt > 0.75:
    print('El modelo de árbol de decisión supera el umbral de exactitud!')
else:
    print('El modelo de árbol de decisión no supera el umbral de exactitud y debe despreciarse')

Datos para modelo de Árbol de decisión:

profundidad del árbol: 4

Exactitud del módelo: 0.7916018662519441
El modelo de árbol de decisión supera el umbral de exactitud!


### Observaciones:
Luego de entrenar el modelo bajo el esquema de árbol de decisión, se encuentra que un árbol con una profundidad de 4, da una exactitud del 79%.


## Modelo de Bosque Aleatorio:

In [7]:
#Definición hiperparámetros iniciales:
best_depth_rf = 0
best_acc_rf = 0
best_est_rf = 0

#Aplicación del modelo
#Aplicando a numero de arboles entre 5 y 30 variando cada 5
for est in range(5, 31, 5):
    #Aplicando a produndidad de arboles entre 1 y 10
    for depth in range(1,11):
        model_rf = RandomForestClassifier(random_state=792604, n_estimators=est, max_depth=depth)
        model_rf.fit(feature_train, target_train)
        prediction_rf = model_rf.predict(feature_valid)
        acc_rf = accuracy_score(target_valid, prediction_rf)

        #Definición de mejores parámetros
        if acc_rf > best_acc_rf:
            best_depth_rf = depth
            best_acc_rf = acc_rf
            best_est_rf = est

#Datos finales
print('Datos para modelo de Bosque aleatorio:')
print()
print('Número de arboles:', best_est_rf)
print()
print('profundidad del árbol:', best_depth_rf)
print()
print('Exactitud del módelo:', best_acc_rf)
if best_acc_rf > 0.75:
    print('El modelo de bosque aleatorio supera el umbral de exactitud!')
else:
    print('El modelo de bosque aleatorio no supera el umbral de exactitud y debe descartarse')

Datos para modelo de Bosque aleatorio:

Número de arboles: 10

profundidad del árbol: 9

Exactitud del módelo: 0.7978227060653188
El modelo de bosque aleatorio supera el umbral de exactitud!


### Observaciones:

El resultado de entrenamiento del modelo de bosque aleatorio ha sido satisfactorio, con un bosque de 10 arboles con 9 elementos de profundidad, se obtiene un 79.78% de exactitud.

## Modelo de Regresión Logística:

In [9]:
#Aplicación del modelo
model_lr = LogisticRegression(random_state=792604, solver='liblinear')
model_lr.fit(feature_train, target_train)
prediction_lr = model_lr.predict(feature_valid)
acc_lr = accuracy_score(target_valid, prediction_lr)

#Datos finales
print('Exactitud del modelo:', acc_lr)

if acc_lr > 0.75:
    print('El modelo de regresión logística supera el umbral de exactitud!')
else:
    print('El modelo de regresión logística no supera el umbral de exactitud y no debe usarse')

Exactitud del modelo: 0.6842923794712286
El modelo de regresión logística no supera el umbral de exactitud y no debe usarse


### Observaciones:

El entrenamiento del modelo de regresión logística tuvo una exactitud del 68.4% y por lo tanto no satisface el umbral de exactitud puesto y no debería tenerse en cuenta.

## Evaluación final:

In [44]:
#Serie de resultados]:
results = pd.DataFrame({'Árbol de decisión': [best_acc_dt], 'Bosque Aleatorio': [best_acc_rf], 'Regresión logística': [acc_lr]})

#Calculo de la exactitud mas alta
best_acc = results.max().max()
print(f'El modelo con la mayor exactitud fue el modelo de: {results.max().idxmax()} con una exactitud del: {best_acc*100:.2f}%')

best_model_name = results.max().idxmax()

if best_model_name == 'Árbol de decisión':
    print(f'Este modelo tiene una profundidad de: {best_depth_dt} unidades de profundidad')
elif best_model_name == 'Bosque Aleatorio':
    print(f'Este modelo tiene {best_est_rf} árboles y {best_depth_rf} unidades de profundidad') 


El modelo con la mayor exactitud fue el modelo de: Bosque Aleatorio con una exactitud del: 79.78%
Este modelo tiene 10 árboles y 9 unidades de profundidad


### Observaciones:

Se realiza un comparativo de exactitudes de los 3 modelos entrenados y se encuentra que el modelo de Bosque aleatorio con 10 arboles con una profundidad de 9 unidades presentó la exactitud mas alta con un 79.78% de exactitud.

## Evaluación de la calidad del modelo  con el grupo de prueba:

In [45]:
#Aplicación del modelo:
final_model = RandomForestClassifier(random_state=792604, n_estimators=best_est_rf, max_depth=best_depth_rf)
final_model.fit(feature_train, target_train)
prediction = final_model.predict(feature_test)
accuracy = accuracy_score(target_test, prediction)

#Exactitud de predicciones aplicadas a Feature_train
prediction_train = final_model.predict(feature_train)
accuracy_train = accuracy_score(target_train, prediction_train)

print(f'La exactitud del modelo aplicado al grupo de prueba fue de: {accuracy*100:.2f}%')

print(f'La exactitud del modelo aplicado al grupo de entrenamiento fue de: {accuracy_train*100:.2f}%')

La exactitud del modelo aplicado al grupo de prueba fue de: 84.14%
La exactitud del modelo aplicado al grupo de entrenamiento fue de: 86.88%


### Observaciones:

- Se realiza la evaluación del modelo en el grupo de prueba, obteniendo una exactitud del 84%. Este resultado es ligeramente mayor al obtenido en la validación del modelo, pero la diferencia es menor al 5%.
- De igual forma, se evalua la exactitud del modelo al tratar de predecir resultados a partir del grupo de entrenamiento, alcanzando una exactitud del 86.88%. Este valor y el obtenido con el grupo de prueba indican que el modelo opera bien con datos nuevos y que el modelo es robusto y no presenta un alto sobreajuste.
- Esta diferencia podría estar relacionada con la aleatoriedad utilizada al segregar los datos de validación y de prueba, pero sin embargo, al ser menor al 5% implica que el modelo entrenado sigue operando bien para predecir el plan a recomendar.


## Prueba de cordura:

In [43]:
from sklearn.dummy import DummyClassifier

dummy = DummyClassifier(strategy='most_frequent', random_state=792604)
dummy.fit(feature_train, target_train)
dummy_pred = dummy.predict(feature_test)
dummy_acc = accuracy_score(target_test, dummy_pred)
print(f'Exactitud del modelo dummy: {dummy_acc*100:.2f}%')

Exactitud del modelo dummy: 73.25%


### Observaciones:

- Al realizar la prueba de cordura se obtiene una exactitud del 73%, lo que significa una diferencia del 6.53% respecto a nuestro modelo.
- Esto muestra que el modelo seleccionado predice mejor que modelos frecuentes aplicados a esta clase de datos. En terminos mas didacticos, nuestro modelo tendrá exito en 4 de cada 5 predicciones, mientras el otro modelo tendría una menor tasa de éxito en cerca de 7 de cada 10 predicciones.


## Conclusiones finales:
- Se ha realizado una continuación al análisis iniciado en sprints anteriores para el análisis de planes utilizados por los clientes de la compañía Megaline.
- Se tomó un juego de datos del comportamiento de clientes asociados a planes Smart y Ultra, este dataset fue revisado previamente para garantizar que no tuviera valores ausentes o duplicados, permitiendo que la aplicación de modelos de entrenamiento fuera robusta.
- Se encuentra que el porcentaje de clientes que utilizan el plan Ultra es cercana al 31%. Aunque es una diferencia amplia, no representa una diferencia definitiva para la aplicación del modelo entrenado.
- Teniendo en cuenta que se tienen 3214 registros, y al no disponer de mas información, además de que se iba a realizar el entrenamiento de un modelo categórico, se decide hacer una segregación de los datos en grupos de entrenamiento, validación y prueba.
- Sobre estos datos evaluamos la implementación de modelos de árbol de decisión, bosque aleatorio y regresión logística, encontrando que el bosque aleatorio de 10 arboles y 9 unidades de profundidad presentaba la mejor exactitud en los resultados.
- Al aplicar la prueba del modelo, esta exactitud difiere en menos del 5% de los resultados de validación, permitiendo corroborar que el modelo sigue dando buenas predicciones.
- Se realiza una prueba de cordura que arroja una exactitud del 73.25% que representa una diferencia mayor al 6%. Esta diferencia implica que nuestro modelo sigue presentando solidez en la predicción de planes a recomendar.
- Finalmente, los datos de exactitud de este modelo implican que funcionará en aproximadamente 4 de cada 5 casos, lo cual es un dato importante y confiable.